# Prompt Tuning

In [10]:
import sys
!{sys.executable} -m pip install -U \
langchain \
langchain_community \
langchain_openai \
openai \
python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/997.1 kB ? eta -:--:--Requirement already satisfied: orjson>=3.9.14 in /Users/austinsand/workspace/context-studio/local-server/nlp_sandbox/.sandbox-venv/lib/python3.13/site-packages (from langsmith>=0.1.17->langchain) (3.11.3)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.1/997.1 kB 32.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.1/997.1 kB 32.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain_openai]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain_openai]


## 01 - Exploration

In [13]:
import os
from dotenv import load_dotenv
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import init_chat_model

# Load environment variables from .env
load_dotenv()
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# Set up the chat model for OpenRouter DeepSeek
llm = None
if True:
    #llm = ChatOpenAI(
    #  openai_api_base="https://openrouter.ai/api/v1",
    #  openai_api_key=openrouter_api_key,
    #  model_name="openai/gpt-5-mini"  # or "deepseek/deepseek-chat-v3.1:free",
    #)
    llm = init_chat_model("openai/gpt-5-mini", model_provider="openai", openai_api_base="https://openrouter.ai/api/v1", openai_api_key=openrouter_api_key)
else:
    llm = init_chat_model("gpt-4o", model_provider="openai", temperature=0)


# Define the prompt pipeline
system_prompt = "You are an expert assistant helping with prompt engineering."
user_prompt_template = "Generate a prompt for a {task} that uses {technique} and targets {audience}."

prompt = ChatPromptTemplate.from_messages([
  SystemMessagePromptTemplate.from_template(system_prompt),
  HumanMessagePromptTemplate.from_template(user_prompt_template)
])

# Example variables
variables = {
  "task": "text summarization",
  "technique": "few-shot learning",
  "audience": "AI researchers"
}

# Format the prompt
messages = prompt.format_messages(**variables)

# Use the new invoke API (avoids BaseChatModel.__call__ deprecation) and extract text safely
response = llm.invoke(messages)

# LangChain response shapes vary across versions; extract text robustly
content = None
if hasattr(response, "generations"):
    # response.generations is often List[List[Generation]]
    try:
        content = response.generations[0][0].text
    except Exception:
        content = str(response)
elif hasattr(response, "output_text"):
    content = response.output_text
elif hasattr(response, "content"):
    # fall back to older attribute
    content = response.content
else:
    content = str(response)

print(content)

Below is a ready-to-use few-shot prompt you can feed to a large language model to produce concise, technical summaries tailored for AI researchers. It includes instructions, three demonstration input→output examples, and a final template to apply to new texts.

Instruction:
You are an assistant that summarizes AI research texts for expert readers. Produce a concise paragraph (1–3 sentences) capturing the core idea and significance, followed by a short structured list of key points. Use precise technical language, preserve important formulas/symbols, cite reported metrics and datasets when present, and explicitly mention limitations, failure modes, and reproducibility details (code, seeds, compute) if given. If a piece of information is not provided in the source, state "Not specified." Keep the summary focused on novelty, method, results, and implications for follow-up research. Target length: ~80–150 words total.

Output format (strict):
Summary: <1–3 sentence summary in technical ton